In [1]:
from pprint import pprint

import weaviate
from weaviate.classes.config import Configure

In [2]:
from weaviate.classes.query import Filter

In [3]:
client = weaviate.connect_to_local()

In [4]:
recipes = client.collections.get("Recipes")

In [5]:

print(
    "Total vectorized: "
    f"{recipes.aggregate.over_all(total_count=True).total_count}"
)
query = "dessert"
print(
    f"semantic search result for {query}"
    f"{recipes.query.near_text(query=query, limit=5)}"
)

with_md_filter = recipes.query.near_text(
    query=query,
    filters=Filter.by_property("calories").less_than(500),
    limit=5,
)
pprint(f"calorie filter: {with_md_filter.objects[0]}")

Total vectorized: 522517
semantic search result for dessertQueryReturn(objects=[Object(uuid=_WeaviateUUIDInt('3a93aa6c-1d8f-5eaf-9b63-89301bb0dde8'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'name': 'C.M.P. Dessert', 'description': 'A wonderful, refreshing dessert. Tastes just like a C.M.P. sundae! Prep time does not include refrigeration prior to serving.', 'rating': 4.5, 'review_count': 4.0, 'recipe_id': 34639, 'ingredients': ['flour', 'margarine', 'unsalted peanuts', 'unsalted peanuts', 'peanut butter', 'cream cheese', "confectioners' sugar", 'Cool Whip', 'instant chocolate pudding mix', 'instant vanilla pudding', 'milk'], 'category': 'Dessert', 'carbs': 53.5, 'calories': 684.4, 'keywords': ['Kid Friendly', '< 60 Mins'], 'protein': 11.0, 'fat': 49.1, 'servings': 12.0}, references=None, vector={}, collection='Recipes'), Object(uuid=_WeaviateUUID

In [6]:
import time


def bench(label, fn, n=5):
    fn()  # warm-up, first call pays connection/cache cost
    times = []
    for _ in range(n):
        t = time.perf_counter()
        res = fn()
        times.append((time.perf_counter() - t) * 1000)
    times.sort()
    print(f"{label:<26} median {times[n // 2]:7.1f} ms   hits {len(res.objects)}")
    return res


def names(res):
    for o in res.objects:
        p = o.properties
        print(f"  {p['name'][:45]:<45} {p['calories']:>7} cal  {p['category']}")

In [7]:
q = "spicy chicken dinner"
cheap = Filter.by_property("calories").less_than(500)

bench("vector only", lambda: recipes.query.near_text(query=q, limit=5))
filtered = bench(
    "vector + calories<500",
    lambda: recipes.query.near_text(query=q, filters=cheap, limit=5),
)
bench(
    "vector + 2 filters",
    lambda: recipes.query.near_text(
        query=q,
        filters=cheap & Filter.by_property("category").equal("Chicken"),
        limit=5,
    ),
)
bench("bm25 keyword", lambda: recipes.query.bm25(query=q, limit=5))
bench("hybrid", lambda: recipes.query.hybrid(query=q, limit=5))

print("\ntop filtered results:")
names(filtered)

vector only                median     7.5 ms   hits 5
vector + calories<500      median    15.0 ms   hits 5
vector + 2 filters         median    24.9 ms   hits 5
bm25 keyword               median     2.2 ms   hits 5
hybrid                     median    12.4 ms   hits 5

top filtered results:
  Devilish Chicken                                404.1 cal  Chicken Breast
  Spicy Honey Chicken Drums                       408.4 cal  Chicken
  Spicy Thai Chicken                              295.4 cal  Chicken
  Dijon Broccoli &amp; Chicken                    214.8 cal  Chicken Breast
  Oven Fried Spicy Chicken                        270.6 cal  Chicken


In [8]:
# how much does a picky filter cost? tighter filter = fewer candidates survive
for cal in (100, 300, 800, 100_000):
    f = Filter.by_property("calories").less_than(cal)
    matching = recipes.aggregate.over_all(filters=f, total_count=True).total_count
    bench(
        f"calories<{cal} ({matching} rows)",
        lambda f=f: recipes.query.near_text(query=q, filters=f, limit=5),
    )

print()
# and how much does asking for more results cost?
for k in (1, 10, 100, 500):
    bench(f"limit={k}", lambda k=k: recipes.query.near_text(query=q, limit=k))

calories<100 (60565 rows)  median    14.4 ms   hits 5
calories<300 (247166 rows) median    19.1 ms   hits 5
calories<800 (461937 rows) median    21.0 ms   hits 5
calories<100000 (522513 rows) median    32.7 ms   hits 5

limit=1                    median     5.1 ms   hits 1
limit=10                   median     5.6 ms   hits 10
limit=100                  median    11.6 ms   hits 100
limit=500                  median    24.7 ms   hits 500


In [ ]:
client.close()